# 🏋️ Addon: Weights vs Oversampling vs scale_pos_weight

Understand and compare 3 methods to handle imbalanced datasets:
- Using sample_weight
- Using scale_pos_weight
- Using Oversampling (SMOTE)


In [ ]:

# Install necessary packages if needed
# !pip install xgboost imbalanced-learn


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE


## 1️⃣ Simulate Dataset Again

In [ ]:

# Simulate 1000 background events (label 0), 10 signal events (label 1)
n_background = 1000
n_signal = 10

X_background = np.random.normal(0, 1, (n_background, 2))
X_signal = np.random.normal(2, 1, (n_signal, 2))

X = np.vstack((X_background, X_signal))
y = np.array([0]*n_background + [1]*n_signal)

df = pd.DataFrame(X, columns=['feature1', 'feature2'])
df['Label'] = y

# Define weights
df['Weight'] = df['Label'].apply(lambda x: 1/100 if x == 1 else 1)

# Split
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, df['Weight'].values, test_size=0.3, random_state=42, stratify=y)

# Report imbalance
print("Train class distribution:", pd.Series(y_train).value_counts().to_dict())


## 2️⃣ Train with sample_weight

In [ ]:

clf_sample_weight = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
clf_sample_weight.fit(X_train, y_train, sample_weight=w_train)

y_pred_sample_weight = clf_sample_weight.predict(X_test)

print("WITH sample_weight:")
print(classification_report(y_test, y_pred_sample_weight))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_sample_weight))


## 3️⃣ Train with scale_pos_weight

In [ ]:

s = np.sum(w_train[y_train == 1])
b = np.sum(w_train[y_train == 0])
scale_pos_weight = b / s
print("scale_pos_weight = ", scale_pos_weight)

clf_scale_pos = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                              scale_pos_weight=scale_pos_weight, random_state=42)
clf_scale_pos.fit(X_train, y_train)

y_pred_scale_pos = clf_scale_pos.predict(X_test)

print("WITH scale_pos_weight:")
print(classification_report(y_test, y_pred_scale_pos))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_scale_pos))


## 4️⃣ Train with Oversampling (SMOTE)

In [ ]:

# Use SMOTE to oversample the minority class
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Check new class distribution
unique, counts = np.unique(y_train_smote, return_counts=True)
print("After SMOTE:", dict(zip(unique, counts)))

clf_smote = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
clf_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = clf_smote.predict(X_test)

print("WITH SMOTE Oversampling:")
print(classification_report(y_test, y_pred_smote))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_smote))



## 📝 Summary of Methods

| Method            | Adds Rows? | Changes Class Ratio? | Overfitting Risk | Training Time | Competition-friendly |
|-------------------|------------|---------------------|------------------|---------------|---------------------|
| `sample_weight`   | ❌ No      | Only in loss        | Low              | Normal        | ✅ Yes              |
| `scale_pos_weight`| ❌ No      | In loss (global)    | Low              | Normal        | ✅ Yes              |
| SMOTE (Oversampling)| ✅ Yes   | In dataset itself   | High (if duplicated) | Higher    | ❌ Not used in Higgs |

👉 For the Higgs Boson challenge: **weights are the correct solution**.

👉 SMOTE is useful for certain models but can lead to overfitting on small datasets.

👉 `sample_weight` and `scale_pos_weight` give more stable and honest modeling.
